# Driver Distraction Detector (phone + eyes off road)

Two independent alerts are raised from a video:

1. **EYES OFF ROAD**: from head pose + **iris tracking** (gaze direction) + eye-closure. Flags when the driver's eyes are not on the road for a sustained time.
2. **PHONE**: YOLOv8 detects a phone; flagged if the head/eyes are off the road or the phone is held near the face.

Everything is calibrated automatically from the first ~2 seconds of the video (assumed: driver looking at the road).

Runtime: *Runtime > Change runtime type > T4 GPU*. Then **Runtime > Run all**.

In [ ]:
!pip -q install ultralytics mediapipe
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

In [ ]:
import cv2, time, base64, subprocess, os, csv
import numpy as np
from collections import deque
from dataclasses import dataclass
from ultralytics import YOLO
import mediapipe as mp
from IPython.display import HTML, display
from google.colab import files

@dataclass
class Config:
    # --- phone ---
    phone_conf: float = 0.25
    yolo_weights: str = "yolov8s.pt"
    near_face_scale: float = 1.6
    phone_window_sec: float = 1.5
    phone_trigger_ratio: float = 0.6
    # --- head pose (deg away from calibrated 'road' pose) ---
    pitch_thresh: float = 18.0
    yaw_thresh: float = 30.0
    # --- eyes / gaze (deviation from calibrated 'road' gaze) ---
    gaze_h_thresh: float = 0.12     # horizontal iris offset (fraction of eye width)
    gaze_v_thresh: float = 0.20     # vertical iris offset (fraction of eye opening)
    ear_closed: float = 0.12        # eye-aspect-ratio below this = eyes closed
    eye_window_sec: float = 1.0     # eyes must be off-road for most of this window
    eye_trigger_ratio: float = 0.7
    # --- general ---
    calib_sec: float = 2.0
    min_flag_sec: float = 0.8       # ignore shorter events in the report

CFG = Config()
PHONE_CLASS_ID = 67  # COCO 'cell phone'

In [ ]:
# ---------- geometry helpers ----------
MODEL_POINTS = np.array([
    (0.0, 0.0, 0.0), (0.0, -330.0, -65.0),
    (-225.0, 170.0, -135.0), (225.0, 170.0, -135.0),
    (-150.0, -150.0, -125.0), (150.0, -150.0, -125.0)], dtype=np.float64)
LM_IDX = [1, 152, 33, 263, 61, 291]

# MediaPipe landmark ids: (outer corner, inner corner, upper lid, lower lid, iris centre)
EYE_R = (33, 133, 159, 145, 468)   # subject's right eye
EYE_L = (263, 362, 386, 374, 473)  # subject's left eye


def head_pose(lms, w, h):
    pts2d = np.array([(lms[i].x * w, lms[i].y * h) for i in LM_IDX], dtype=np.float64)
    cam = np.array([[w, 0, w / 2], [0, w, h / 2], [0, 0, 1]], dtype=np.float64)
    ok, rvec, _ = cv2.solvePnP(MODEL_POINTS, pts2d, cam, np.zeros((4, 1)), flags=cv2.SOLVEPNP_ITERATIVE)
    if not ok:
        return None, None
    rmat, _ = cv2.Rodrigues(rvec)
    pitch, yaw, roll = cv2.RQDecomp3x3(rmat)[0]
    if pitch > 90: pitch -= 180
    elif pitch < -90: pitch += 180
    xs = [l.x * w for l in lms]; ys = [l.y * h for l in lms]
    return (pitch, yaw), (min(xs), min(ys), max(xs), max(ys))


def eye_metrics(lms, w, h):
    # returns gaze_h, gaze_v, ear, iris pixel points
    def P(i): return np.array([lms[i].x * w, lms[i].y * h])
    hs, vs, ears, iris = [], [], [], []
    for k, (o, i_, u, l, c) in enumerate((EYE_R, EYE_L)):
        o, i_, u, l, c = P(o), P(i_), P(u), P(l), P(c)
        width = np.linalg.norm(i_ - o) + 1e-6
        r = float(np.dot(c - o, i_ - o) / width**2)       # 0=outer corner, 1=inner corner
        hs.append(r if k == 0 else 1 - r)                  # make both eyes move the same way
        vs.append(float((c[1] - u[1]) / max(l[1] - u[1], 1e-6)))
        ears.append(float(np.linalg.norm(l - u) / width))
        iris.append(tuple(c))
    return np.mean(hs), np.mean(vs), np.mean(ears), iris

In [ ]:
class DistractionDetector:
    def __init__(self, fps, cfg=CFG):
        self.cfg, self.fps = cfg, fps
        self.yolo = YOLO(cfg.yolo_weights)
        from mediapipe.tasks import python as mp_python
        from mediapipe.tasks.python import vision
        opts = vision.FaceLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path="face_landmarker.task"),
            running_mode=vision.RunningMode.VIDEO, num_faces=1)
        self.face = vision.FaceLandmarker.create_from_options(opts)
        self.ph_hist = deque(maxlen=max(1, int(fps * cfg.phone_window_sec)))
        self.eye_hist = deque(maxlen=max(1, int(fps * cfg.eye_window_sec)))
        self.calib_frames = int(fps * cfg.calib_sec)
        self.calib_buf, self.base = [], None   # base = (pitch, yaw, gaze_h, gaze_v)

    def _phones(self, frame):
        r = self.yolo(frame, classes=[PHONE_CLASS_ID], conf=self.cfg.phone_conf, verbose=False)[0]
        return [tuple(map(float, b)) for b in r.boxes.xyxy.cpu().numpy()]

    def process(self, frame, ts_ms):
        c = self.cfg
        h, w = frame.shape[:2]
        rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        res = self.face.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), int(ts_ms))
        phones = self._phones(frame)

        face_found = len(res.face_landmarks) > 0
        face_box, iris, dev = None, [], None
        head_off = gaze_off = closed = False

        if face_found:
            lms = res.face_landmarks[0]
            pose, face_box = head_pose(lms, w, h)
            gh, gv, ear, iris = eye_metrics(lms, w, h)
            closed = ear < c.ear_closed
            if pose is not None:
                cur = (pose[0], pose[1], gh, gv)
                if self.base is None and not closed:
                    self.calib_buf.append(cur)
                    if len(self.calib_buf) >= self.calib_frames:
                        self.base = tuple(np.median(self.calib_buf, axis=0))
                b = self.base if self.base else cur
                dp, dy, dgh, dgv = cur[0] - b[0], cur[1] - b[1], cur[2] - b[2], cur[3] - b[3]
                dev = (dp, dy, dgh, dgv)
                head_off = abs(dp) > c.pitch_thresh or abs(dy) > c.yaw_thresh
                gaze_off = (not closed) and (abs(dgh) > c.gaze_h_thresh or abs(dgv) > c.gaze_v_thresh)

        # ---------- EYES-OFF-ROAD ----------
        if not face_found:
            eyes_bad, eyes_reason = True, "Face not visible"
        elif closed:
            eyes_bad, eyes_reason = True, "Eyes closed"
        elif head_off:
            eyes_bad, eyes_reason = True, "Head turned away from road"
        elif gaze_off:
            eyes_bad, eyes_reason = True, "Eyes looking away from road"
        else:
            eyes_bad, eyes_reason = False, ""
        self.eye_hist.append(eyes_bad)
        eyes_flag = (len(self.eye_hist) == self.eye_hist.maxlen and
                     sum(self.eye_hist) / len(self.eye_hist) >= c.eye_trigger_ratio)

        # ---------- PHONE ----------
        near_face = False
        if face_box is not None:
            fx, fy = (face_box[0] + face_box[2]) / 2, (face_box[1] + face_box[3]) / 2
            fw = (face_box[2] - face_box[0]) * c.near_face_scale
            fh = (face_box[3] - face_box[1]) * c.near_face_scale
            for (x1, y1, x2, y2) in phones:
                if abs((x1 + x2) / 2 - fx) < fw and abs((y1 + y2) / 2 - fy) < fh:
                    near_face = True
        phone = len(phones) > 0
        ph_bad = phone and (head_off or gaze_off or near_face or not face_found)
        ph_reason = ""
        if ph_bad:
            ph_reason = "Phone held near face" if near_face and not (head_off or gaze_off) else "Using phone (eyes/head off road)"
        self.ph_hist.append(ph_bad)
        phone_flag = (len(self.ph_hist) == self.ph_hist.maxlen and
                      sum(self.ph_hist) / len(self.ph_hist) >= c.phone_trigger_ratio)

        return dict(phones=phones, face_box=face_box, iris=iris, dev=dev,
                    calibrated=self.base is not None,
                    eyes_flag=eyes_flag, eyes_reason=eyes_reason,
                    phone_flag=phone_flag, phone_reason=ph_reason)


def draw(frame, info):
    out = frame.copy()
    H, W = out.shape[:2]
    for (x1, y1, x2, y2) in info["phones"]:
        cv2.rectangle(out, (int(x1), int(y1)), (int(x2), int(y2)), (0, 165, 255), 2)
        cv2.putText(out, "phone", (int(x1), int(y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)
    if info["face_box"] is not None:
        fb = info["face_box"]
        cv2.rectangle(out, (int(fb[0]), int(fb[1])), (int(fb[2]), int(fb[3])), (255, 200, 0), 1)
    for (x, y) in info["iris"]:                       # show tracked irises
        cv2.circle(out, (int(x), int(y)), 3, (0, 255, 0), -1)
    if not info["calibrated"]:
        txt = "calibrating (look at the road)..."
    elif info["dev"] is not None:
        dp, dy, dgh, dgv = info["dev"]
        txt = f"head p{dp:+.0f} y{dy:+.0f} | gaze h{dgh:+.2f} v{dgv:+.2f}"
    else:
        txt = "no face"
    cv2.putText(out, txt, (10, H - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    y = 30
    if info["eyes_flag"]:
        cv2.rectangle(out, (0, 0), (W - 1, H - 1), (0, 0, 255), 6)
        cv2.putText(out, "EYES OFF ROAD: " + info["eyes_reason"], (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        y += 30
    if info["phone_flag"]:
        cv2.putText(out, "PHONE: " + info["phone_reason"], (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 140, 255), 2)
    return out

In [ ]:
def run_on_video(path, out_path="annotated_raw.mp4"):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    det = DistractionDetector(fps)

    events = []                                     # (type, start, end, reason)
    open_ev = {"EYES OFF ROAD": None, "PHONE": None}
    n, t0 = 0, time.time()
    while True:
        ok, frame = cap.read()
        if not ok: break
        info = det.process(frame, n * 1000.0 / fps)
        writer.write(draw(frame, info))
        t = n / fps
        for kind, flag, reason in (("EYES OFF ROAD", info["eyes_flag"], info["eyes_reason"]),
                                   ("PHONE", info["phone_flag"], info["phone_reason"])):
            if flag and open_ev[kind] is None:
                open_ev[kind] = [t, reason]
            elif not flag and open_ev[kind] is not None:
                s, r = open_ev[kind]
                if t - s >= CFG.min_flag_sec: events.append((kind, s, t, r))
                open_ev[kind] = None
        n += 1
    end_t = n / fps
    for kind, v in open_ev.items():
        if v is not None: events.append((kind, v[0], end_t, v[1]))
    cap.release(); writer.release()
    print(f"Processed {n} frames in {time.time()-t0:.1f}s ({n/max(time.time()-t0,1e-6):.1f} FPS)")

    final = "annotated.mp4"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", out_path, "-vcodec", "libx264",
                    "-pix_fmt", "yuv420p", final])
    return final, sorted(events, key=lambda e: e[1])


def show_video(path, width=640):
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width={width} controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

## Analyse a video
- `SOURCE = "upload"`: a file picker appears
- `SOURCE = "drive"`: reads `VIDEO_PATH` from Google Drive (better for big files)

The first ~2 s of the video are used as the "looking at the road" reference.

In [ ]:
# ================== SETTINGS ==================
SOURCE = "upload"                                   # "upload" or "drive"
VIDEO_PATH = "/content/drive/MyDrive/my_video.mp4"  # only for SOURCE == "drive"
# ==============================================

if SOURCE == "upload":
    uploaded = files.upload()
    video_path = list(uploaded.keys())[0]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    video_path = VIDEO_PATH
assert os.path.exists(video_path), f"Video not found: {video_path}"

result_path, events = run_on_video(video_path)

print("\n=== FLAGGED EVENTS ===")
if not events: print("Nothing flagged.")
for kind, s, e, r in events:
    print(f"[{kind:13s}] {s:6.1f}s -> {e:6.1f}s ({e-s:.1f}s)  {r}")

with open("report.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["type", "start_sec", "end_sec", "duration_sec", "reason"])
    for kind, s, e, r in events: w.writerow([kind, round(s, 2), round(e, 2), round(e - s, 2), r])

show_video(result_path)
files.download(result_path)
files.download("report.csv")

## Tuning
- Green dots on the eyes = iris tracking is working. The bottom line shows live deviation from the "road" pose.
- **Too many false alarms** -> raise `gaze_h_thresh` / `gaze_v_thresh` / `yaw_thresh`, or `eye_window_sec`.
- **Misses** -> lower those thresholds.
- Low-res footage (480p) makes iris tracking noisy, so head pose carries most of the signal; higher-res, front-facing footage gives better gaze results.
- If the driver is *not* looking at the road during the first 2 s, calibration will be wrong. Trim the clip or increase `calib_sec` accordingly.